# 06 — Multivariable Cox PH: time until deterioration (primary)

**Primary:** conditional median remaining years, within observed training event-time support only. No hazard extrapolation is used for the output.

In [2]:

from pathlib import Path
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if PROJECT.name == "bridge_time_primary_notebooks":
    PROJECT = PROJECT.parent
FINAL = PROJECT / "data" / "final"
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

train_data = pd.read_parquet(
    FINAL / "bridge_survival_train.parquet"
)

test_data = pd.read_parquet(
    FINAL / "bridge_survival_test_aligned.parquet"
)

print("Training records:", f"{len(train_data):,}")
print("Testing records:", f"{len(test_data):,}")
print("Training events:", f"{train_data['event'].sum():,}")
print("Testing events:", f"{test_data['event'].sum():,}")


Training records: 498,875
Testing records: 171,766
Training events: 237,197
Testing events: 22,396


## 2. Data Preprocessing

In [3]:

# Select bridge characteristics for the multivariable Cox model.
numeric_predictors = [
    "bridge_age",
    "ADT_029",
    "STRUCTURE_LEN_MT_049"
]

categorical_predictors = [
    "STRUCTURE_KIND_043A",
    "STRUCTURE_TYPE_043B"
]

predictor_columns = numeric_predictors + categorical_predictors

# Prepare training and testing datasets.
cox_train = train_data[
    predictor_columns + ["duration", "event"]
].copy()

cox_test = test_data[
    ["bridge_id", "entry_duration"] + predictor_columns
].copy()

print("Training shape:", cox_train.shape)
print("Testing shape:", cox_test.shape)
print("Predictors:", predictor_columns)


Training shape: (498875, 7)
Testing shape: (171766, 7)
Predictors: ['bridge_age', 'ADT_029', 'STRUCTURE_LEN_MT_049', 'STRUCTURE_KIND_043A', 'STRUCTURE_TYPE_043B']


In [4]:

# Fill missing values using statistics from the training data.
for column in numeric_predictors:
    median_value = cox_train[column].median()
    cox_train[column] = cox_train[column].fillna(median_value)
    cox_test[column] = cox_test[column].fillna(median_value)

for column in categorical_predictors:
    mode_value = cox_train[column].mode()[0]
    cox_train[column] = cox_train[column].fillna(mode_value)
    cox_test[column] = cox_test[column].fillna(mode_value)

# Check for remaining missing values.
print("Training missing values:", cox_train.isna().sum().sum())
print("Testing missing values:", cox_test.isna().sum().sum())


Training missing values: 0
Testing missing values: 0


In [5]:

# Encode categorical predictors using training categories.
for column in categorical_predictors:
    categories = sorted(cox_train[column].unique())
    cox_train[column] = pd.Categorical(
        cox_train[column], categories=categories
    )
    cox_test[column] = pd.Categorical(
        cox_test[column], categories=categories
    )

# Convert categories into numerical dummy variables.
cox_train = pd.get_dummies(
    cox_train,
    columns=categorical_predictors,
    drop_first=True,
    dtype=int
)

cox_test = pd.get_dummies(
    cox_test,
    columns=categorical_predictors,
    drop_first=True,
    dtype=int
)

# Ensure both datasets use identical predictor columns.
training_predictors = cox_train.drop(
    columns=["duration", "event"]
).columns

cox_test = cox_test.reindex(
    columns=["bridge_id", "entry_duration"] + training_predictors.tolist(),
    fill_value=0
)

print("Training shape:", cox_train.shape)
print("Testing shape:", cox_test.shape)
print("Number of predictors:", len(training_predictors))
print("Predictor columns match:",
      cox_train[training_predictors].columns.equals(
          cox_test[training_predictors].columns
      ))


Training shape: (498875, 36)
Testing shape: (171766, 36)
Number of predictors: 34
Predictor columns match: True


In [6]:

from sklearn.preprocessing import StandardScaler

# Standardise numerical predictors using training statistics.
scaler = StandardScaler()

cox_train[numeric_predictors] = scaler.fit_transform(
    cox_train[numeric_predictors]
)

cox_test[numeric_predictors] = scaler.transform(
    cox_test[numeric_predictors]
)

# Check the standardised training predictors.
print("Training means:")
print(cox_train[numeric_predictors].mean().round(4))

print("\nTraining standard deviations:")
print(cox_train[numeric_predictors].std().round(4))

print("\nRemaining missing values:")
print("Training:", cox_train.isna().sum().sum())
print("Testing:", cox_test.isna().sum().sum())


Training means:
bridge_age             -0.0
ADT_029                 0.0
STRUCTURE_LEN_MT_049   -0.0
dtype: float64

Training standard deviations:
bridge_age              1.0
ADT_029                 1.0
STRUCTURE_LEN_MT_049    1.0
dtype: float64

Remaining missing values:
Training: 0
Testing: 0


## 3. Model Training

In [7]:

# Initialise the Cox PH model with regularisation.
cox_model = CoxPHFitter(penalizer=0.01)

# Train the model using survival times and deterioration events.
cox_model.fit(
    cox_train,
    duration_col="duration",
    event_col="event"
)

# Display training performance.
print("Multivariable Cox PH Model")
print("-------------------------")
print("Training C-index:", round(cox_model.concordance_index_, 4))
print("Log-likelihood:", round(cox_model.log_likelihood_, 2))
print("Number of predictors:", len(training_predictors))


/Users/annie/Downloads/bridge-deterioration/.venv/lib/python3.12/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['STRUCTURE_TYPE_043B_13'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)


Multivariable Cox PH Model
-------------------------
Training C-index: 0.6759
Log-likelihood: -2910221.71
Number of predictors: 34


## 4. Primary objective: conditional median remaining years

The Cox survival curve is a step function. Report `NaN` when it does not reach 50% conditional survival before the last training event time. This avoids unsupported extrapolation.

In [8]:
from lifelines.utils import concordance_index
# Step-function baseline cumulative hazard H0(t), with H0(t)=0 before first event.
base = cox_model.baseline_cumulative_hazard_.iloc[:,0]
t_grid = base.index.to_numpy(dtype=float)
h_grid = base.to_numpy(dtype=float)
entry = cox_test['entry_duration'].to_numpy(dtype=float)
relative_hazard = cox_model.predict_partial_hazard(cox_test[training_predictors]).to_numpy().ravel()
# Use right-continuous step lookup; do not interpolate between event times.
def H_at(times):
    ix = np.searchsorted(t_grid, np.asarray(times), side='right')-1
    return np.where(ix >= 0, h_grid[np.maximum(ix,0)], 0.0)
# The median after entry occurs when exp[-r*(H0(t)-H0(entry))] <= 0.5.
# Invert the step curve using searchsorted on monotone cumulative hazard.
target_h = H_at(entry) + np.log(2)/relative_hazard
idx = np.searchsorted(h_grid, target_h, side='left')
supported = (idx < len(t_grid)) & np.isfinite(entry) & (entry >= 0) & (entry <= t_grid[-1])
median_total = np.full(len(entry), np.nan)
median_total[supported] = t_grid[idx[supported]]
remaining = np.maximum(median_total-entry,0)
primary_predictions = test_data[['bridge_id','entry_duration','test_duration','event']].copy()
primary_predictions['median_remaining_years'] = remaining
primary_predictions['median_total_years_since_first_good'] = median_total
primary_predictions['median_estimable'] = supported
primary_predictions['risk_score'] = relative_hazard
primary_predictions.to_csv(RESULTS/'time_cox_multivariable.csv',index=False)
print('Estimable:',int(supported.sum()),'/',len(supported))
print(primary_predictions['median_remaining_years'].describe().to_string())
if supported.sum()>2 and test_data.loc[supported,'event'].sum()>0:
 print('Ranking C-index on estimable subset (NOT directly comparable with other subsets):',round(concordance_index(test_data.loc[supported,'test_duration'],remaining[supported],test_data.loc[supported,'event']),4))
print('Saved:',RESULTS/'time_cox_multivariable.csv')

Estimable: 45961 / 171766
count    45961.000000
mean        16.921412
std          5.153476
min          1.000000
25%         14.000000
50%         16.000000
75%         21.000000
max         28.000000
Ranking C-index on estimable subset (NOT directly comparable with other subsets): 0.6571
Saved: /Users/annie/Downloads/bridge-deterioration/results/time_cox_multivariable.csv


### Full-test Cox ranking evaluation (primary time-to-deterioration analysis)

This uses **all** test bridges and the fitted Cox relative hazard, not just the subset with estimable median remaining times. It evaluates discrimination, **not** numerical accuracy of predicted remaining years. Because testing starts after a landmark time (`entry_duration`), this conventional C-index is a descriptive ranking measure rather than a complete censoring-aware, landmark-adjusted validation.

In [9]:
# Full-test Cox hazard-ranking C-index (all test bridges, not just estimable medians).
# Higher partial hazard means earlier deterioration, so negate the hazard
# for lifelines.concordance_index (which expects larger scores = longer survival).
from lifelines.utils import concordance_index

full_test_relative_hazard = cox_model.predict_partial_hazard(
    cox_test[training_predictors]
).to_numpy().ravel()

full_test_duration = test_data["test_duration"].to_numpy(dtype=float)
full_test_event = test_data["event"].to_numpy(dtype=int)
valid = (
    np.isfinite(full_test_relative_hazard)
    & np.isfinite(full_test_duration)
    & (full_test_duration >= 0)
    & np.isfinite(full_test_event)
)

assert valid.all(), "Missing/invalid test data; investigate before calculating C-index."
assert len(full_test_relative_hazard) == len(test_data)

cox_full_test_c_index = concordance_index(
    full_test_duration,
    -full_test_relative_hazard,
    event_observed=full_test_event
)

print("Full-test Cox hazard-ranking C-index:", round(cox_full_test_c_index, 4))
print("Test bridges evaluated:", len(test_data))
print("Estimable median remaining times:", int(supported.sum()))
print("Important: C-index measures ranking, not error in predicted years.")

full_test_metrics = pd.DataFrame([{
    "model": "Multivariable Cox PH",
    "metric": "full_test_hazard_ranking_c_index",
    "c_index": cox_full_test_c_index,
    "test_samples": len(test_data),
    "median_remaining_years_estimable": int(supported.sum())
}])
full_test_metrics.to_csv(RESULTS / "cox_multivariable_time_metrics.csv", index=False)
print("Saved:", RESULTS / "cox_multivariable_time_metrics.csv")


Full-test Cox hazard-ranking C-index: 0.5923
Test bridges evaluated: 171766
Estimable median remaining times: 45961
Important: C-index measures ranking, not error in predicted years.
Saved: /Users/annie/Downloads/bridge-deterioration/results/cox_multivariable_time_metrics.csv


In [10]:
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

# Use a 50,000-bridge sample so the test runs quickly (the full 498,875 is slow).
sub = cox_train.sample(50000, random_state=42)
m = CoxPHFitter(penalizer=0.01).fit(sub, duration_col="duration", event_col="event")

ph = proportional_hazard_test(m, sub, time_transform="rank")
print(ph.summary)

/Users/annie/Downloads/bridge-deterioration/.venv/lib/python3.12/site-packages/lifelines/utils/__init__.py:1100: ConvergenceWarning: Column(s) ['STRUCTURE_TYPE_043B_13', 'STRUCTURE_TYPE_043B_14', 'STRUCTURE_TYPE_043B_15', 'STRUCTURE_TYPE_043B_18'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)


                        test_statistic             p   -log2(p)
ADT_029                       0.146821  7.015916e-01   0.511297
STRUCTURE_KIND_043A_1         3.007735  8.286802e-02   3.593041
STRUCTURE_KIND_043A_2         0.023325  8.786143e-01   0.186698
STRUCTURE_KIND_043A_3         0.929500  3.349928e-01   1.577798
STRUCTURE_KIND_043A_4         0.720175  3.960866e-01   1.336112
STRUCTURE_KIND_043A_5         0.884901  3.468628e-01   1.527563
STRUCTURE_KIND_043A_6        11.166261  8.329835e-04  10.229424
STRUCTURE_KIND_043A_7         0.272597  6.015950e-01   0.733135
STRUCTURE_KIND_043A_8         2.344764  1.257046e-01   2.991891
STRUCTURE_KIND_043A_9         0.308444  5.786368e-01   0.789270
STRUCTURE_LEN_MT_049          2.089697  1.482955e-01   2.753453
STRUCTURE_TYPE_043B_01        0.442839  5.057553e-01   0.983489
STRUCTURE_TYPE_043B_02        0.741478  3.891878e-01   1.361461
STRUCTURE_TYPE_043B_03        0.117785  7.314499e-01   0.451169
STRUCTURE_TYPE_043B_04        0.005082  